
# 02 — Preprocessing Axe 3 (Version 2)


## 1. Problèmes identifiés dans la V1

### Limites de la Version 1

La première version du preprocessing a permis de construire une base fonctionnelle pour la modélisation. Cependant, plusieurs limites ont été identifiées :

### 1. Déséquilibre des classes
La distribution des classes montre que la **classe 0 est largement majoritaire par rapport à la classe 1**.  

**Impact :**
- biais des modèles vers la classe majoritaire ;
- certains modèles, comme Random Forest, peuvent ignorer une grande partie des décès ;
- recall très faible sur certaines approches.

### 2. Imputation trop simple
La Version 1 utilisait une imputation simple basée sur :
- la **médiane** pour les variables numériques ;
- le **mode** pour les variables catégorielles.

**Problème :**
- perte d’information clinique potentielle ;
- stratégie trop basique pour des données médicales complexes.

### 3. Pas de Feature Engineering
Dans la Version 1, les variables étaient utilisées de manière relativement brute.

Or, le cahier des charges souligne l’importance d’un **score RDEF** construit à partir des déficits neurologiques, ainsi que l’usage de l’état de conscience `RCONSC` dans la logique clinique de construction des variables.

### 4. Encodage basique
L’encodage One-Hot direct appliqué sans structure complète peut introduire :
- du bruit ;
- de la redondance ;
- une moindre maîtrise du pipeline global.

### 5. Pas de pipeline structuré
La Version 1 ne reposait pas sur un pipeline complet et centralisé.

**Conséquences :**
- risque de data leakage si les traitements sont mal séquencés ;
- difficulté de reproductibilité ;
- maintenance moins propre du projet.

---

## 2. Objectif du Preprocessing V2

Construire un pipeline **robuste** et **médicalement cohérent** pour préparer les données de l’Axe 3.

---

## 3. Ce qu’on va améliorer

### 1. Feature Engineering (IMPORTANT)
- Construction du **score RDEF**
- Transformation de **RCONSC**

### 2. Pipeline sklearn propre
- Imputation + encodage intégrés dans un pipeline structuré

### 3. Gestion du déséquilibre
- **NE PAS** la faire ici directement
- Elle sera réalisée dans le notebook de **modeling V2** avec **SMOTE**

### 4. Sélection des variables intelligente
- Conservation des variables cliniquement pertinentes
- Réduction du bruit et du risque de fuite d’information



## Objectif général du notebook

Ce notebook prépare les données du dataset IST pour la **modélisation de l’Axe 3**, c’est-à-dire la **prédiction de la mortalité post-AVC** :

- `DDEAD` : mortalité à **14 jours**
- `FDEAD` : mortalité à **6 mois**

La Version 2 améliore la robustesse du preprocessing en introduisant :
- du **feature engineering** ;
- une meilleure **sélection des variables** ;
- un **pipeline sklearn propre** ;
- une préparation plus cohérente pour la phase de modélisation.


In [5]:

# =========================
# 1. IMPORTATION DES LIBRAIRIES
# =========================
# Cette section importe les bibliothèques nécessaires
# au chargement, au nettoyage et à la préparation
# avancée des données dans Google Colab.

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Options d'affichage
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)



## Chargement du dataset depuis Google Drive




In [6]:

# =========================
# 2. CHARGEMENT DU DATASET
# =========================

drive.mount('/content/drive')

path = "/content/drive/MyDrive/Stroke_Project_ML/Data/IST_corrected.csv"

df = pd.read_csv(path, encoding="latin1")

print("Dataset chargé avec succès.")
print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")

df.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset chargé avec succès.
Nombre de lignes : 19435
Nombre de colonnes : 112


,HOSPNUM,RDELAY,RCONSC,SEX,AGE,RSLEEP,RATRIAL,RCT,RVISINF,RHEP24,RASP3,RSBP,RDEF1,RDEF2,RDEF3,RDEF4,RDEF5,RDEF6,RDEF7,RDEF8,STYPE,RDATE,HOURLOCAL,MINLOCAL,DAYLOCAL,RXASP,RXHEP,DASP14,DASPLT,DLH14,DMH14,DHH14,ONDRUG,DSCH,DIVH,DAP,DOAC,DGORM,DSTER,DCAA,DHAEMD,DCAREND,DTHROMB,DMAJNCH,DMAJNCHD,DMAJNCHX,DSIDE,DSIDED,DSIDEX,DDIAGISC,DDIAGHA,DDIAGUN,DNOSTRK,DNOSTRKX,DRSISC,DRSISCD,DRSH,DRSHD,DRSUNK,DRSUNKD,DPE,DPED,DALIVE,DALIVED,DPLACE,DDEAD,DDEADD,DDEADC,DDEADX,FDEAD,FLASTD,FDEADD,FDEADC,FDEADX,FRECOVER,FDENNIS,FPLACE,FAP,FOAC,FU1_RECD,FU2_DONE,COUNTRY,CNTRYNUM,FU1_COMP,NCCODE,CMPLASP,CMPLHEP,DIED,TD,EXPDD,EXPD6,EXPD14,SET14D,ID14,OCCODE,DEAD1,DEAD2,DEAD3,DEAD4,DEAD5,DEAD6,DEAD7,DEAD8,H14,ISC14,NK14,STRK14,HTI14,PE14,DVT14,TRAN14,NCB14
0,1,17,D,M,69,Y,NaN,Y,Y,NaN,NaN,140,N,N,N,Y,N,Y,N,Y,PACS,sty-91,99,99,4,Y,N,Y,Y,N,NaN,N,14.0,NaN,NaN,N,N,N,N,N,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,N,NaN,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,N,Y,E,NaN,NaN,14.0,187.0,UK,27,NaN,13,Y,Y,0,187.0,0.6980,0.2344,0.1054,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,10,F,M,76,Y,NaN,Y,N,NaN,NaN,150,Y,Y,Y,N,N,N,N,N,LACS,sty-91,99,99,7,N,L,N,Y,Y,NaN,N,14.0,NaN,NaN,N,N,N,N,Y,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,N,NaN,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,N,Y,A,NaN,NaN,16.0,192.0,UK,27,NaN,NaN,Y,Y,0,192.0,0.5389,0.1555,0.0421,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,43,F,F,71,N,NaN,Y,N,NaN,NaN,170,Y,Y,Y,N,N,N,N,N,LACS,sty-91,99,99,3,Y,N,Y,Y,N,NaN,N,11.0,NaN,NaN,N,N,N,N,N,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,Y,11.0,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,N,Y,A,NaN,NaN,11.0,189.0,UK,27,NaN,NaN,Y,Y,0,189.0,0.5275,0.1009,0.0323,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,6,F,M,81,N,NaN,N,N,NaN,NaN,170,N,N,N,Y,N,N,N,N,PACS,sty-91,99,99,7,N,H,N,Y,N,NaN,Y,14.0,NaN,NaN,N,N,N,N,Y,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,Y,16.0,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,Y,N,A,NaN,NaN,23.0,183.0,UK,27,NaN,NaN,Y,Y,0,183.0,0.4021,0.1147,0.0244,1,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4,20,F,M,78,N,NaN,N,N,NaN,NaN,170,Y,Y,Y,N,N,N,N,N,LACS,lut-91,99,99,6,Y,H,Y,Y,N,NaN,Y,14.0,NaN,NaN,N,N,N,N,N,N,NaN,NaN,N,NaN,NaN,N,NaN,NaN,Y,N,N,N,NaN,N,NaN,N,NaN,NaN,NaN,N,NaN,N,NaN,NaN,N,NaN,0.0,NaN,N,NaN,NaN,NaN,NaN,N,Y,E,NaN,NaN,17.0,214.0,UK,27,NaN,NaN,Y,Y,0,214.0,0.5600,0.1709,0.0441,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



## Vérification initiale

Avant le preprocessing, on vérifie :
- la structure globale du dataset ;
- la présence des variables cibles ;
- la disponibilité des colonnes cliniques utiles à l’Axe 3.


In [7]:

# =========================
# 3. VERIFICATION GENERALE
# =========================

print("DDEAD présente ?", "DDEAD" in df.columns)
print("FDEAD présente ?", "FDEAD" in df.columns)

print("\nAperçu des colonnes :")
display(pd.DataFrame({"colonne": df.columns}))


DDEAD présente ? True
FDEAD présente ? True

Aperçu des colonnes :


,colonne
0,HOSPNUM
1,RDELAY
2,RCONSC
3,SEX
4,AGE
5,RSLEEP
6,RATRIAL
7,RCT
8,RVISINF
9,RHEP24



## Nettoyage général du dataset

Cette étape standardise le dataset avant le feature engineering.

On applique ici :
1. nettoyage des noms de colonnes ;
2. remplacement de certains faux codes de valeurs manquantes ;
3. harmonisation du format général.


In [8]:

# =========================
# 4. NETTOYAGE GENERAL
# =========================

# Nettoyage des noms de colonnes
df.columns = df.columns.str.strip()

# Standardisation de quelques faux codes de valeurs manquantes
df = df.replace(["?", "??", "NA", "N/A", ""], np.nan)

print("Nettoyage général effectué.")
print("\nValeurs manquantes par colonne (top 20) :")
display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("nb_valeurs_manquantes"))


Nettoyage général effectué.

Valeurs manquantes par colonne (top 20) :


,nb_valeurs_manquantes
FLASTD,19377
DRSHD,19338
DPED,19310
DMAJNCHX,19290
DMAJNCHD,19286
DDEADX,19267
DRSUNKD,19175
DRSISCD,19022
DNOSTRKX,19008
DSIDED,18802



## Sélection intelligente des variables pour l’Axe 3

Dans cette Version 2, on conserve un ensemble de variables à la fois :
- **cliniquement pertinentes** ;
- **disponibles précocement** ;
- **compatibles avec la prédiction de la mortalité**.

### Variables retenues
- `AGE`
- `SEX`
- `RCONSC`
- `RSBP`
- `RATRIAL`
- `RDELAY`
- `RDEF1` à `RDEF8`
- `STYPE`
- `RXASP`
- `RXHEP`

### Variables cibles
- `DDEAD`
- `FDEAD`

Ce choix permet d’éviter certaines variables tardives susceptibles d’introduire du **data leakage**.


In [9]:

# =========================
# 5. SELECTION DES VARIABLES
# =========================

selected_columns = [
    "AGE", "SEX", "RCONSC", "RSBP", "RATRIAL", "RDELAY",
    "RDEF1", "RDEF2", "RDEF3", "RDEF4", "RDEF5", "RDEF6", "RDEF7", "RDEF8",
    "STYPE", "RXASP", "RXHEP",
    "DDEAD", "FDEAD"
]

df_model = df[selected_columns].copy()

print("Shape après sélection :", df_model.shape)
df_model.head()


Shape après sélection : (19435, 19)


,AGE,SEX,RCONSC,RSBP,RATRIAL,RDELAY,RDEF1,RDEF2,RDEF3,RDEF4,RDEF5,RDEF6,RDEF7,RDEF8,STYPE,RXASP,RXHEP,DDEAD,FDEAD
0,69,M,D,140,NaN,17,N,N,N,Y,N,Y,N,Y,PACS,Y,N,N,N
1,76,M,F,150,NaN,10,Y,Y,Y,N,N,N,N,N,LACS,N,L,N,N
2,71,F,F,170,NaN,43,Y,Y,Y,N,N,N,N,N,LACS,Y,N,N,N
3,81,M,F,170,NaN,6,N,N,N,Y,N,N,N,N,PACS,N,H,N,N
4,78,M,F,170,NaN,20,Y,Y,Y,N,N,N,N,N,LACS,Y,H,N,N



## Feature Engineering

La Version 2 introduit deux transformations importantes.

### 1. Construction du score `RDEF_SCORE`
Le score `RDEF_SCORE` est construit à partir des colonnes `RDEF1` à `RDEF8`.

Principe :
- `Y` → 1
- `N` → 0

Puis on calcule la somme des déficits neurologiques.

### 2. Transformation de `RCONSC`
La variable `RCONSC` reflète l’état de conscience à l’admission.

Transformation utilisée :
- `F` → 0
- `D` → 1
- `U` → 2

Cette transformation permet une représentation plus exploitable du niveau de conscience dans les modèles.


In [10]:

# =========================
# 6. FEATURE ENGINEERING
# =========================

# Colonnes de déficit neurologique
rdef_cols = ["RDEF1", "RDEF2", "RDEF3", "RDEF4", "RDEF5", "RDEF6", "RDEF7", "RDEF8"]

# Conversion des variables RDEF en format binaire
for col in rdef_cols:
    df_model[col] = df_model[col].astype(str).str.strip().str.upper()
    df_model[col] = df_model[col].replace({
        "Y": 1,
        "N": 0,
        "U": np.nan,
        "NAN": np.nan,
        "": np.nan
    })
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# Construction du score RDEF
df_model["RDEF_SCORE"] = df_model[rdef_cols].sum(axis=1, min_count=1)

# Transformation de RCONSC
df_model["RCONSC"] = df_model["RCONSC"].astype(str).str.strip().str.upper()
mapping_rconsc = {"F": 0, "D": 1, "U": 2}
df_model["RCONSC_NUM"] = df_model["RCONSC"].replace(mapping_rconsc)
df_model["RCONSC_NUM"] = pd.to_numeric(df_model["RCONSC_NUM"], errors="coerce")

print("Feature engineering terminé.")
display(df_model[["RDEF_SCORE", "RCONSC", "RCONSC_NUM"]].head())


Feature engineering terminé.


,RDEF_SCORE,RCONSC,RCONSC_NUM
0,3.0,D,1
1,3.0,F,0
2,3.0,F,0
3,1.0,F,0
4,3.0,F,0



## Préparation des cibles `DDEAD` et `FDEAD`

Pour chaque cible :
- `Y` → 1
- `N` → 0
- `U` / valeurs inconnues → `NaN`

Ensuite, on supprime uniquement les lignes où la cible étudiée est manquante.
On prépare ainsi deux datasets distincts :
- un pour `DDEAD`
- un pour `FDEAD`


In [11]:

# =========================
# 7. PREPARATION DES CIBLES
# =========================

def clean_target(dataframe, target_col):
    df_temp = dataframe.copy()

    df_temp[target_col] = df_temp[target_col].astype(str).str.strip().str.upper()

    mapping_target = {
        "Y": 1,
        "N": 0,
        "U": np.nan,
        "NAN": np.nan,
        "": np.nan
    }

    df_temp[target_col] = df_temp[target_col].replace(mapping_target)
    df_temp[target_col] = pd.to_numeric(df_temp[target_col], errors="coerce")

    df_temp = df_temp.dropna(subset=[target_col]).copy()

    return df_temp

df_ddead = clean_target(df_model, "DDEAD")
df_fdead = clean_target(df_model, "FDEAD")

print("Distribution DDEAD après nettoyage :")
print(df_ddead["DDEAD"].value_counts(dropna=False))

print("\nDistribution FDEAD après nettoyage :")
print(df_fdead["FDEAD"].value_counts(dropna=False))


Distribution DDEAD après nettoyage :
DDEAD
0.0    17376
1.0     2034
Name: count, dtype: int64

Distribution FDEAD après nettoyage :
FDEAD
0.0    14910
1.0     4369
Name: count, dtype: int64



## Construction des jeux `X` et `y`

On prépare séparément les données pour :
- la prédiction de `DDEAD`
- la prédiction de `FDEAD`

On retire des features :
- la cible à prédire ;
- l’autre cible ;
- les colonnes devenues redondantes après feature engineering si nécessaire.


In [12]:

# =========================
# 8. SEPARATION FEATURES / TARGET
# =========================

# Variables retenues après feature engineering
final_features = [
    "AGE", "SEX", "RSBP", "RATRIAL", "RDELAY",
    "STYPE", "RXASP", "RXHEP",
    "RDEF_SCORE", "RCONSC_NUM"
]

X_ddead = df_ddead[final_features].copy()
y_ddead = df_ddead["DDEAD"].astype(int)

X_fdead = df_fdead[final_features].copy()
y_fdead = df_fdead["FDEAD"].astype(int)

print("Shape X_ddead :", X_ddead.shape)
print("Shape y_ddead :", y_ddead.shape)

print("\nShape X_fdead :", X_fdead.shape)
print("Shape y_fdead :", y_fdead.shape)


Shape X_ddead : (19410, 10)
Shape y_ddead : (19410,)

Shape X_fdead : (19279, 10)
Shape y_fdead : (19279,)



## Définition des colonnes numériques et catégorielles

Pour construire un pipeline propre, on sépare :
- les variables numériques ;
- les variables catégorielles.

### Variables numériques
- `AGE`
- `RSBP`
- `RDELAY`
- `RDEF_SCORE`
- `RCONSC_NUM`

### Variables catégorielles
- `SEX`
- `RATRIAL`
- `STYPE`
- `RXASP`
- `RXHEP`


In [13]:

# =========================
# 9. DEFINITION DES TYPES DE VARIABLES
# =========================

num_cols = ["AGE", "RSBP", "RDELAY", "RDEF_SCORE", "RCONSC_NUM"]
cat_cols = ["SEX", "RATRIAL", "STYPE", "RXASP", "RXHEP"]

print("Variables numériques :", num_cols)
print("Variables catégorielles :", cat_cols)


Variables numériques : ['AGE', 'RSBP', 'RDELAY', 'RDEF_SCORE', 'RCONSC_NUM']
Variables catégorielles : ['SEX', 'RATRIAL', 'STYPE', 'RXASP', 'RXHEP']



## Construction du pipeline de preprocessing

Ici, on met en place un pipeline structuré avec `scikit-learn`.

### Partie numérique
- imputation par la médiane

### Partie catégorielle
- imputation par la modalité la plus fréquente
- encodage avec `OneHotEncoder(handle_unknown="ignore")`

Cette structure est plus propre, plus reproductible et plus sûre qu’une suite de transformations manuelles.


In [14]:

# =========================
# 10. PIPELINE DE PREPROCESSING
# =========================

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

print("Pipeline de preprocessing créé avec succès.")
preprocessor


Pipeline de preprocessing créé avec succès.


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['AGE', 'RSBP', 'RDELAY', 'RDEF_SCORE',
                                  'RCONSC_NUM']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['SEX', 'RATRIAL', 'STYPE', 'RXASP',
                                  'RXHEP'])])


## Transformation finale des données

Le pipeline est appliqué séparément sur :
- le dataset `DDEAD`
- le dataset `FDEAD`

Pour faciliter la réutilisation dans le notebook de modeling, on reconvertit ensuite les sorties en DataFrames.


In [15]:

# =========================
# 11. TRANSFORMATION DES DONNEES
# =========================

# Transformation pour DDEAD
X_ddead_processed = preprocessor.fit_transform(X_ddead)

# Récupération des noms de colonnes après transformation
feature_names_ddead = preprocessor.get_feature_names_out()
X_ddead_processed = pd.DataFrame(
    X_ddead_processed.toarray() if hasattr(X_ddead_processed, "toarray") else X_ddead_processed,
    columns=feature_names_ddead
)

# Transformation pour FDEAD
X_fdead_processed = preprocessor.fit_transform(X_fdead)

feature_names_fdead = preprocessor.get_feature_names_out()
X_fdead_processed = pd.DataFrame(
    X_fdead_processed.toarray() if hasattr(X_fdead_processed, "toarray") else X_fdead_processed,
    columns=feature_names_fdead
)

print("Transformation terminée.")
print("Shape X_ddead_processed :", X_ddead_processed.shape)
print("Shape X_fdead_processed :", X_fdead_processed.shape)


Transformation terminée.
Shape X_ddead_processed : (19410, 20)
Shape X_fdead_processed : (19279, 20)



## Vérifications finales

Avant de sauvegarder les fichiers preprocessés, on vérifie :
- qu’il ne reste plus de valeurs manquantes ;
- que les targets sont bien binaires ;
- que les dimensions sont cohérentes.


In [16]:

# =========================
# 12. VERIFICATIONS FINALES
# =========================

print("NaN restants dans X_ddead_processed :", X_ddead_processed.isna().sum().sum())
print("NaN restants dans X_fdead_processed :", X_fdead_processed.isna().sum().sum())

print("\nDistribution finale y_ddead :")
print(y_ddead.value_counts())

print("\nDistribution finale y_fdead :")
print(y_fdead.value_counts())


NaN restants dans X_ddead_processed : 0
NaN restants dans X_fdead_processed : 0

Distribution finale y_ddead :
DDEAD
0    17376
1     2034
Name: count, dtype: int64

Distribution finale y_fdead :
FDEAD
0    14910
1     4369
Name: count, dtype: int64



## Sauvegarde des données preprocessées

Les fichiers sont exportés vers Google Drive pour être utilisés dans le notebook de **Modeling V2**.

### Fichiers sauvegardés
- `X_ddead_preprocessed_v2.csv`
- `y_ddead_preprocessed_v2.csv`
- `X_fdead_preprocessed_v2.csv`
- `y_fdead_preprocessed_v2.csv`


In [17]:

# =========================
# 13. SAUVEGARDE DES FICHIERS
# =========================

save_dir = "/content/drive/MyDrive/Stroke_Project_ML/Data/"

X_ddead_processed.to_csv(save_dir + "X_ddead_preprocessed_v2.csv", index=False)
y_ddead.to_csv(save_dir + "y_ddead_preprocessed_v2.csv", index=False)

X_fdead_processed.to_csv(save_dir + "X_fdead_preprocessed_v2.csv", index=False)
y_fdead.to_csv(save_dir + "y_fdead_preprocessed_v2.csv", index=False)

print("Fichiers V2 sauvegardés avec succès dans Google Drive.")


Fichiers V2 sauvegardés avec succès dans Google Drive.



## Conclusion

Cette **Version 2 du preprocessing** améliore significativement la Version 1.

### Améliorations apportées
- prise en compte explicite des limites observées en V1 ;
- ajout d’un **feature engineering clinique** ;
- meilleure sélection des variables ;
- création d’un **pipeline sklearn structuré** ;
- préparation plus robuste pour la modélisation.

### Étape suivante
La prochaine étape consiste à construire le **Modeling V2**, en intégrant :
- une gestion du déséquilibre avec **SMOTE** ;
- des modèles plus performants ;
- une optimisation orientée **Recall**.
